# Silver layer walkthrough

This notebook reads the latest matching rows from `bronze_coin_gecko` and `bronze_currency_rate`, parses their raw JSON payloads, and writes normalized coin prices to `silver_coin_prices`.

## 1. Imports and database connection

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

repo_root = Path.cwd()
if not (repo_root / '.env').exists():
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

def required_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Missing required environment variable: {name}')
    return value

engine = create_engine(
    URL.create(
        drivername='postgresql+psycopg2',
        username=required_env('DB_USER'),
        password=required_env('DB_PASSWORD'),
        host=required_env('DB_HOST'),
        port=int(os.getenv('DB_PORT', '5432')),
        database=required_env('DB_NAME'),
    ),
    connect_args={'sslmode': 'require'},
)

## 2. Read the latest CoinGecko bronze row

In [ ]:
coin_bronze = pd.read_sql(
    text('''
        SELECT batch_id, extracted_at, raw_payload
        FROM bronze_coin_gecko
        ORDER BY extracted_at DESC
        LIMIT 1
    '''),
    engine,
)

if coin_bronze.empty:
    raise RuntimeError('No CoinGecko bronze data found. Run bronze first.')

batch_id = coin_bronze.loc[0, 'batch_id']
extracted_at = coin_bronze.loc[0, 'extracted_at']
batch_id

## 3. Read the currency row for the same batch

In [ ]:
currency_bronze = pd.read_sql(
    text('''
        SELECT batch_id, raw_payload
        FROM bronze_currency_rate
        WHERE batch_id = :batch_id
        ORDER BY extracted_at DESC
        LIMIT 1
    '''),
    engine,
    params={'batch_id': batch_id},
)

if currency_bronze.empty:
    raise RuntimeError(f'No currency bronze data found for batch {batch_id}.')

print('Matching batch:', currency_bronze.loc[0, 'batch_id'] == batch_id)

## 4. Parse both raw payloads

In [ ]:
def payload_as_dict(payload):
    return payload if isinstance(payload, dict) else json.loads(payload)

coin_payload = payload_as_dict(coin_bronze.loc[0, 'raw_payload'])
currency_payload = payload_as_dict(currency_bronze.loc[0, 'raw_payload'])

print('Coins:', list(coin_payload))
print('Currency date:', currency_payload.get('date'))

## 5. Normalize coin prices and calculate IDR

In [ ]:
coins = pd.DataFrame([
    {'coin_id': coin_id, 'price_usd': values['usd']}
    for coin_id, values in coin_payload.items()
])
exchange_rate = currency_payload['rates']['IDR']

silver = coins.assign(
    batch_id=batch_id,
    extracted_at=extracted_at,
    exchange_rate=exchange_rate,
    price_idr=coins['price_usd'] * exchange_rate,
    provider_date=currency_payload.get('date'),
)[['batch_id', 'extracted_at', 'coin_id', 'price_usd', 'exchange_rate', 'price_idr', 'provider_date']]

silver

## 6. Write and validate the silver table

In [ ]:
silver.to_sql('silver_coin_prices', con=engine, if_exists='append', index=False)

stored_silver = pd.read_sql(
    text('SELECT * FROM silver_coin_prices WHERE batch_id = :batch_id ORDER BY coin_id'),
    engine,
    params={'batch_id': batch_id},
)

stored_silver